# MH-01. MH PheCode nodes + cohort + spine skeleton

Broad mental-disorders cohort (NOT psychosis-risk). Locked decisions:

- Cohort = persons with at least one condition mapping to an MH-category PheCode (v1.2),
  AND whose earliest MH dx (index) falls at age 18-35 inclusive. Earlier-than-18 index
  is excluded (index is NOT reset to a later dx).
- index_date = min(condition_start_date) over MH dx. Tie-break: smallest 3-digit parent,
  then smallest full PheCode (numeric).
- ICD-9-CM / ICD-10-CM -> PheCode only. SNOMED-only dx are counted as unmapped (not mapped).
- years_in_ehr_before_index derives from the first/last raw EHR event across the full tables\n  (this release has no observation_period table); single source, reused in MH-04.

Scope cuts (agreed): no FAMD/MFA, no ADI/SVI, no note_nlp, no SNOMED mapping.

Output: `output/mh/spine.parquet` + node list + unmapped counts.


## 1. Config and external file checks

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb

DATA_PATH = '/home/jupyter/2836994-data2'      # same OMOP release as psychosis pipeline
PHECODE_MAP_PATH = 'phecode_icd10.csv'                    # ICD-9 map (existing; misnamed content)
PHECODE_MAP_ICD10_PATH = 'Phecode_map_v1_2_icd10cm_beta.csv'  # official ICD-10-CM map (columns ICD10CM+PHECODE)
PHECODE_DEF_PATH = 'phecode_definitions1.2.csv'  # PheCode -> description + category (confirm exists)
OUTPUT_DIR = Path('output/mh')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Locked cohort parameters
AGE_MIN, AGE_MAX = 18, 35          # inclusive; earlier-than-min index is excluded, not reset
CROSSWALK_VERSION = 'phecode_v1.2' # recorded in provenance

# MH 3-digit parent categories (v1.2 mental disorders)
MH_PARENTS = {295, 296, 300, 301, 303, 304, 305, 306, 313, 316, 317, 318}
PSYCHOSIS_PARENT = 295             # 295.x force-retained downstream; later-psychosis outcome

con = duckdb.connect()
print('DuckDB:', duckdb.__version__)
for p in [PHECODE_MAP_PATH, PHECODE_DEF_PATH]:
    print(f'{p:40s} exists = {Path(p).exists()}')
print('DATA_PATH:', DATA_PATH)


## 2. Build MH PheCode node set from the crosswalk

In [ ]:
# Merge BOTH phecode maps: the ICD-9 map (phecode_icd10.csv, misnamed but ICD-9 content) and
# the official ICD-10-CM map (Phecode_map_v1_2_icd10cm_beta.csv, columns ICD10CM + PHECODE).
# The data is ICD-10-CM dominant, so the ICD-10 map is what unlocks the bulk of diagnoses.
def load_map(path, icd_hint=None):
    d = pd.read_csv(path, dtype=str)
    d.columns = [c.strip() for c in d.columns]
    lower = {c.lower(): c for c in d.columns}
    icd_c = next((lower[k] for k in lower if 'icd' in k), d.columns[0])
    phe_c = next((lower[k] for k in lower if 'phe' in k and 'excl' not in k.lower()), None)
    phe_c = phe_c or next(c for c in d.columns if 'phe' in c.lower())
    out = d[[icd_c, phe_c]].rename(columns={icd_c: 'icd', phe_c: 'phecode'}).dropna()
    out['icd'] = out['icd'].str.strip().str.upper()
    out['phecode'] = out['phecode'].str.strip()
    out['vocab'] = icd_hint
    return out

maps = []
if Path(PHECODE_MAP_PATH).exists():
    maps.append(load_map(PHECODE_MAP_PATH, 'ICD9'))       # existing ICD-9 map
if Path(PHECODE_MAP_ICD10_PATH).exists():
    maps.append(load_map(PHECODE_MAP_ICD10_PATH, 'ICD10')) # official ICD-10-CM map
if not maps:
    raise FileNotFoundError('No phecode map found (need ICD-9 and/or ICD-10-CM csv).')

phe_map = pd.concat(maps, ignore_index=True).drop_duplicates(subset=['icd', 'phecode'])
print('merged crosswalk rows:', len(phe_map),
      '| by vocab:', phe_map['vocab'].value_counts().to_dict())

def parent3(phecode):
    try:
        return int(float(phecode))
    except Exception:
        return np.nan

phe_map['parent3'] = phe_map['phecode'].map(parent3)
mh_map = phe_map[phe_map['parent3'].isin(MH_PARENTS)].copy()
mh_nodes = sorted(mh_map['phecode'].unique(), key=lambda s: float(s))
print(f'MH crosswalk rows: {len(mh_map):,} | distinct MH nodes: {len(mh_nodes)}')
print('MH map by vocab:', mh_map['vocab'].value_counts().to_dict())

# PheCode labels + category
if Path(PHECODE_DEF_PATH).exists():
    defs = pd.read_csv(PHECODE_DEF_PATH, dtype=str)
    defs.columns = [c.strip().lower() for c in defs.columns]
    label_col = next((c for c in ['phenotype','description','phecode_str'] if c in defs.columns), None)
    node_labels = (defs[['phecode', label_col]].rename(columns={label_col:'label'})
                   if label_col else pd.DataFrame(columns=['phecode','label']))
else:
    print('WARNING: PHECODE_DEF_PATH missing -> labels blank, category from parent3 only')
    node_labels = pd.DataFrame({'phecode': mh_nodes, 'label': ['']*len(mh_nodes)})

node_table = (pd.DataFrame({'phecode': mh_nodes})
              .assign(parent3=lambda d: d['phecode'].map(parent3))
              .merge(node_labels, on='phecode', how='left'))
node_table['is_psychosis'] = node_table['parent3'] == PSYCHOSIS_PARENT
node_table.to_parquet(OUTPUT_DIR / 'mh_node_table.parquet', index=False)
print('Saved node table:', node_table.shape)
node_table.head()


## 3. Map each patient's diagnoses to MH PheCodes (ICD-9/10 only)

In [ ]:
# Vocabulary-aware matching (strict) + SNOMED bridge, all inside the enclave (no external files).
#   route 1: ICD-10-CM source_value -> ICD-10 phecode map
#   route 2: ICD-9-CM  source_value -> ICD-9  phecode map
#   route 3: SNOMED rows -> concept_relationship('Maps to') -> standard ICD-10-CM concept_code
#            -> ICD-10 phecode map   (bridge is lossy: SNOMED->ICD10 is many-to-many / may break)
#   fallback: rows with no source vocab -> merged map on source_value
# Union of routes, deduped to distinct (person, date, phecode).
def to_dotted(s):
    s = s.astype('string').str.upper().str.strip()
    no_dot = ~s.str.contains(r'\\.', na=False) & (s.str.len() > 3)
    return s.mask(no_dot, s.str.slice(0, 3) + '.' + s.str.slice(3))

mh_map = mh_map.copy()
mh_map['icd_dot'] = to_dotted(mh_map['icd'])
con.register('mh_map_icd9',  mh_map[mh_map['vocab']=='ICD9'][['icd_dot','phecode','parent3']])
con.register('mh_map_icd10', mh_map[mh_map['vocab']=='ICD10'][['icd_dot','phecode','parent3']])
con.register('mh_map_all',   mh_map[['icd_dot','phecode','parent3']].drop_duplicates())
phe_all = phe_map[['icd']].copy(); phe_all['icd_dot'] = to_dotted(phe_all['icd'])
con.register('phe_all', phe_all[['icd_dot']].drop_duplicates())

def gp(table):
    return f"read_parquet('{DATA_PATH}/{table}/*.parquet')"

DOTIFY = (lambda col:
    f"CASE WHEN POSITION('.' IN {col}) > 0 OR LENGTH({col}) <= 3 THEN UPPER(TRIM({col})) "
    f"ELSE UPPER(TRIM(SUBSTR({col},1,3))) || '.' || UPPER(TRIM(SUBSTR({col},4))) END")

# SNOMED -> ICD-10-CM bridge from concept_relationship('Maps to'), pre-built as a lookup:
# snomed_concept_id -> dotted ICD-10-CM code. Standard OMOP columns concept_id_1/2, relationship_id.
con.sql(f'''
CREATE OR REPLACE TEMP VIEW snomed_to_icd10 AS
SELECT cr.concept_id_1 AS snomed_concept_id,
       {DOTIFY('tgt.concept_code')} AS icd10_dot
FROM {gp('concept_relationship')} cr
JOIN {gp('concept')} src ON cr.concept_id_1 = src.concept_id AND src.vocabulary_id = 'SNOMED'
JOIN {gp('concept')} tgt ON cr.concept_id_2 = tgt.concept_id AND tgt.vocabulary_id = 'ICD10CM'
WHERE cr.relationship_id = 'Mapped from'   -- SNOMED (std) 'Mapped from' ICD10CM source; see note below
''')
# NOTE: direction of 'Maps to' vs 'Mapped from' depends on which side is standard. In OMOP,
# ICD10CM(source) --'Maps to'--> SNOMED(standard). To go SNOMED->ICD10CM we read the reverse
# ('Mapped from'). If the bridge count below is ~0, flip to relationship_id = 'Maps to' and swap
# the vocabulary_id filters on src/tgt.

# Per-row source code + vocab + (for SNOMED rows) the mapped ICD10 code.
con.sql(f'''
CREATE OR REPLACE TEMP VIEW cond_vocab AS
SELECT c.person_id,
       COALESCE(c.condition_start_date, CAST(c.condition_start_datetime AS DATE)) AS dx_date,
       {DOTIFY('c.condition_source_value')} AS code_dot,
       co.vocabulary_id AS vocab,
       c.condition_source_concept_id AS src_cid,
       c.condition_concept_id AS std_cid
FROM {gp('condition_occurrence')} c
LEFT JOIN {gp('concept')} co ON c.condition_source_concept_id = co.concept_id
WHERE COALESCE(c.condition_start_date, CAST(c.condition_start_datetime AS DATE)) IS NOT NULL
''')

mh_dx = con.sql('''
SELECT DISTINCT person_id, dx_date, phecode, parent3 FROM (
    SELECT v.person_id, v.dx_date, m.phecode, m.parent3
    FROM cond_vocab v JOIN mh_map_icd10 m ON v.code_dot = m.icd_dot
    WHERE v.vocab = 'ICD10CM'
  UNION ALL
    SELECT v.person_id, v.dx_date, m.phecode, m.parent3
    FROM cond_vocab v JOIN mh_map_icd9 m ON v.code_dot = m.icd_dot
    WHERE v.vocab = 'ICD9CM'
  UNION ALL
    -- SNOMED bridge: standard concept -> ICD10 -> MH phecode
    SELECT v.person_id, v.dx_date, m.phecode, m.parent3
    FROM cond_vocab v
    JOIN snomed_to_icd10 b ON v.std_cid = b.snomed_concept_id
    JOIN mh_map_icd10 m ON b.icd10_dot = m.icd_dot
    WHERE v.vocab = 'SNOMED' OR v.vocab IS NULL
  UNION ALL
    SELECT v.person_id, v.dx_date, m.phecode, m.parent3
    FROM cond_vocab v JOIN mh_map_all m ON v.code_dot = m.icd_dot
    WHERE v.vocab IS NULL OR v.vocab NOT IN ('ICD9CM','ICD10CM','SNOMED')
)
''').to_df()
mh_dx['dx_date'] = pd.to_datetime(mh_dx['dx_date'])

# --- Coverage diagnostics: how much does the SNOMED bridge actually recover? ---
bridge_cov = con.sql('''
SELECT
  (SELECT COUNT(*) FROM cond_vocab WHERE vocab='SNOMED') AS snomed_rows,
  (SELECT COUNT(*) FROM cond_vocab v JOIN snomed_to_icd10 b ON v.std_cid=b.snomed_concept_id
     WHERE v.vocab='SNOMED') AS snomed_bridged_to_icd10,
  (SELECT COUNT(*) FROM cond_vocab v JOIN snomed_to_icd10 b ON v.std_cid=b.snomed_concept_id
     JOIN mh_map_icd10 m ON b.icd10_dot=m.icd_dot WHERE v.vocab='SNOMED') AS snomed_into_mh
''').to_df()
print('SNOMED bridge coverage (rows):')
print(bridge_cov)

route = con.sql('''
SELECT vocab, COUNT(*) AS n_rows
FROM cond_vocab v WHERE EXISTS (SELECT 1 FROM mh_map_all m WHERE m.icd_dot=v.code_dot)
   OR EXISTS (SELECT 1 FROM snomed_to_icd10 b JOIN mh_map_icd10 m ON b.icd10_dot=m.icd_dot
              WHERE b.snomed_concept_id=v.std_cid)
GROUP BY vocab ORDER BY n_rows DESC
''').to_df()
print('MH candidate rows by source vocab (incl. bridged SNOMED):')
print(route)
print('MH-mapped dx rows:', f'{len(mh_dx):,}', '| persons:', f"{mh_dx['person_id'].nunique():,}")

unmapped = con.sql('''
SELECT COUNT(*) AS n_rows, COUNT(DISTINCT person_id) AS n_persons
FROM cond_vocab v
LEFT JOIN phe_all p ON v.code_dot = p.icd_dot
LEFT JOIN snomed_to_icd10 b ON v.std_cid = b.snomed_concept_id
WHERE p.icd_dot IS NULL AND b.snomed_concept_id IS NULL
''').to_df()
print('Rows still not mapping to ANY phecode:')
print(unmapped)


## 4. Index date, tie-break, and 18-35 age gate

In [ ]:
# Attach birth_datetime for age_at_index.
persons = con.sql(f"SELECT person_id, birth_datetime, gender_source_value FROM {gp('person')}").to_df()
persons['birth_datetime'] = pd.to_datetime(persons['birth_datetime'])

# Earliest MH dx date per person = candidate index.
idx_date = mh_dx.groupby('person_id', as_index=False)['dx_date'].min().rename(columns={'dx_date':'index_date'})

# Tie-break among dx on the index date: smallest parent3, then smallest full phecode.
tie = mh_dx.merge(idx_date, on='person_id')
tie = tie[tie['dx_date'] == tie['index_date']].copy()
tie['phe_num'] = tie['phecode'].astype(float)
tie = tie.sort_values(['person_id','parent3','phe_num'])
index_pick = tie.groupby('person_id', as_index=False).first()[['person_id','index_date','parent3','phecode']]
index_pick = index_pick.rename(columns={'parent3':'index_signal_type','phecode':'index_phecode'})

spine = index_pick.merge(persons, on='person_id', how='left')
spine['age_at_index'] = ((spine['index_date'] - spine['birth_datetime']).dt.days // 365.25).astype('Int64')

before = len(spine)
spine = spine[(spine['age_at_index'] >= AGE_MIN) & (spine['age_at_index'] <= AGE_MAX)].copy()
print(f'Persons with MH dx: {before:,} -> after 18-35 index gate: {len(spine):,}')

# Assertions: gate is exactly inclusive and index is the earliest MH dx.
assert spine['age_at_index'].between(AGE_MIN, AGE_MAX).all()
assert (spine.merge(idx_date, on='person_id')['index_date_x']
        .eq(spine.merge(idx_date, on='person_id')['index_date_y']).all())
print('age gate + index assertions passed')


## 5. Spine fields from raw EHR event span (no observation_period table)

In [ ]:
# years_in_ehr_before_index + post_index_observation_days.
# This OMOP release has NO observation_period table, so derive the EHR span directly
# from the raw full event tables: op_start = first ever event, op_end = last ever event.
# This single computed field is the SOLE lookback filter in MH-04.
import glob

EHR_SPAN_TABLES = [
    ('condition_occurrence', 'condition_start_date'),
    ('visit_occurrence',     'visit_start_date'),
    ('measurement',          'measurement_date'),
    ('drug_exposure',        'drug_exposure_start_date'),
    ('procedure_occurrence', 'procedure_date'),
    ('observation',          'observation_date'),
]
present = [(t, c) for (t, c) in EHR_SPAN_TABLES if glob.glob(f'{DATA_PATH}/{t}/*.parquet')]
print('EHR-span tables present:', [t for t, _ in present])
if not present:
    raise FileNotFoundError('No event tables found under DATA_PATH to derive EHR span.')

union_sql = ' UNION ALL '.join(
    f'SELECT person_id, {c} AS d FROM {gp(t)} WHERE {c} IS NOT NULL' for t, c in present
)
op = con.sql(f'''
SELECT person_id, MIN(d) AS op_start, MAX(d) AS op_end
FROM ({union_sql})
GROUP BY person_id
''').to_df()
op['op_start'] = pd.to_datetime(op['op_start'])
op['op_end'] = pd.to_datetime(op['op_end'])

spine = spine.merge(op, on='person_id', how='left')
spine['years_in_ehr_before_index'] = (spine['index_date'] - spine['op_start']).dt.days / 365.25
spine['post_index_observation_days'] = (spine['op_end'] - spine['index_date']).dt.days
spine['years_in_ehr_before_index'] = spine['years_in_ehr_before_index'].clip(lower=0)
spine['post_index_observation_days'] = spine['post_index_observation_days'].clip(lower=0)

# index_concept_label from the node table.
spine = spine.merge(node_table[['phecode','label']].rename(columns={'phecode':'index_phecode',
                    'label':'index_concept_label'}), on='index_phecode', how='left')
spine['sex'] = spine['gender_source_value']

# all_negative / all_masked are filled in MH-02 (need Y); placeholders here.
spine['all_negative'] = pd.NA
spine['all_masked'] = pd.NA

spine = spine.sort_values('person_id').reset_index(drop=True)
spine_cols = ['person_id','index_date','age_at_index','index_signal_type','index_phecode',
              'index_concept_label','sex','years_in_ehr_before_index',
              'post_index_observation_days','all_negative','all_masked']
spine = spine[spine_cols]
spine.to_parquet(OUTPUT_DIR / 'spine.parquet', index=False)
mh_dx.sort_values('person_id').to_parquet(OUTPUT_DIR / 'mh_dx_events.parquet', index=False)

print('spine:', spine.shape)
print('index_signal_type distribution:')
print(spine['index_signal_type'].value_counts().sort_index())
spine.head()


## 6. Provenance stub

In [ ]:
prov = {
    'crosswalk_version': CROSSWALK_VERSION,
    'age_gate': [AGE_MIN, AGE_MAX],
    'n_mh_nodes': len(mh_nodes),
    'n_cohort': int(len(spine)),
    'unmapped_rows': int(unmapped['n_rows'].iloc[0]),
    'unmapped_persons': int(unmapped['n_persons'].iloc[0]),
    'scope_cuts': ['no_FAMD','no_ADI_SVI','no_note_nlp','no_SNOMED_mapping'],
}
import json
(OUTPUT_DIR / 'provenance_mh01.json').write_text(json.dumps(prov, indent=2))
print(json.dumps(prov, indent=2))
